<a href="https://colab.research.google.com/github/nanthini-1990/GitHubWorkshop/blob/main/Customer_Order_Analysis_using_SQL_Aggregation_and_Joins.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import sqlite3

customers_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/customers.csv"
orders_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/orders.csv"

customers_df = pd.read_csv(customers_url)
orders_df = pd.read_csv(orders_url)

conn = sqlite3.connect(":memory:")

customers_df.to_sql("customers", conn, index=False, if_exists="replace")
orders_df.to_sql("orders", conn, index=False, if_exists="replace")

print("Data Loaded Successfully")

Data Loaded Successfully


In [ ]:
# Task 1 — Aggregation and Grouping
query1 = """
SELECT
    CustomerID,
    COUNT(OrderID) AS order_count,
    SUM(Freight) AS total_freight,
    AVG(Freight) AS avg_freight
FROM orders
GROUP BY CustomerID
ORDER BY total_freight DESC;
"""

result1 = pd.read_sql_query(query1, conn)
result1.head(10)

,customerID,order_count,total_freight,avg_freight
0,SAVEA,31,6683.70,215.603226
1,ERNSH,30,6205.39,206.846333
2,QUICK,28,5605.63,200.201071
3,HUNGO,19,2755.24,145.012632
4,RATTC,18,2134.21,118.567222
5,QUEEN,13,1982.70,152.515385
6,FOLKO,19,1678.08,88.320000
7,BERGS,18,1559.52,86.640000
8,FRANK,15,1403.44,93.562667
9,MEREP,13,1394.22,107.247692


In [ ]:
# Task 2 — WHERE vs HAVING
# Query A
queryA = """
SELECT
    CustomerID,
    COUNT(OrderID) AS high_freight_orders
FROM orders
WHERE Freight > 50
GROUP BY CustomerID;
"""

resultA = pd.read_sql_query(queryA, conn)
resultA.head()

,customerID,high_freight_orders
0,ALFKI,2
1,ANTON,2
2,AROUT,2
3,BERGS,11
4,BLAUS,1


In [ ]:
# Query B
queryB = """
SELECT
    CustomerID,
    SUM(Freight) AS total_freight
FROM orders
GROUP BY CustomerID
HAVING SUM(Freight) > 500;
"""

resultB = pd.read_sql_query(queryB, conn)
resultB.head()

,customerID,total_freight
0,BERGS,1559.52
1,BLONP,623.66
2,BONAP,1357.87
3,BOTTM,793.95
4,EASTC,832.34


Query A uses WHERE, which filters individual rows before aggregation. Only orders with Freight > 50 are considered.

Query B uses HAVING, which filters after aggregation. It considers all orders first, then selects customers whose total freight exceeds 500.

Therefore, Query A filters rows, while Query B filters groups.

In [ ]:
# Task 3 — JOIN and Aggregation
# INNER JOIN (Only customers with orders)
query_inner = """
SELECT
    c.CompanyName,
    c.Country,
    COUNT(o.OrderID) AS order_count,
    SUM(o.Freight) AS total_freight
FROM customers c
INNER JOIN orders o
ON c.CustomerID = o.CustomerID
GROUP BY c.CompanyName, c.Country;
"""

result_inner = pd.read_sql_query(query_inner, conn)
result_inner.head()

,companyName,country,order_count,total_freight
0,Alfreds Futterkiste,Germany,6,225.58
1,Ana Trujillo Emparedados y helados,Mexico,4,97.42
2,Antonio Moreno Taquería,Mexico,7,268.52
3,Around the Horn,UK,13,471.95
4,B's Beverages,UK,10,281.31


In [ ]:
# LEFT JOIN (All customers included)
query_left = """
SELECT
    c.CompanyName,
    c.Country,
    COUNT(o.OrderID) AS order_count,
    SUM(o.Freight) AS total_freight
FROM customers c
LEFT JOIN orders o
ON c.CustomerID = o.CustomerID
GROUP BY c.CompanyName, c.Country;
"""

result_left = pd.read_sql_query(query_left, conn)
result_left.head()

,companyName,country,order_count,total_freight
0,Alfreds Futterkiste,Germany,6,225.58
1,Ana Trujillo Emparedados y helados,Mexico,4,97.42
2,Antonio Moreno Taquería,Mexico,7,268.52
3,Around the Horn,UK,13,471.95
4,B's Beverages,UK,10,281.31


The INNER JOIN includes only customers who have matching records in the orders table (i.e., customers who placed orders).

The LEFT JOIN includes all customers, even those who have not placed any orders. For such customers, total_freight appears as NULL.

The difference occurs because INNER JOIN filters unmatched rows, while LEFT JOIN preserves all records from the left table.